# SSA publication and aggregate coverage

The DHS Program Indicator Data API, The Demographic and Health Surveys (DHS) Program. ICF. Originally funded by the United States Agency for International Development (USAID). Available from https://api.dhsprogram.com/. See fetch_dates for access times. SurveyBridge AI is independent of The DHS Program.

Counts describe retrieved records, not population health trends. Read ssa_summary.md and ssa_comparability_report.md. Report tables are not automatically extracted.

In [ ]:
from pathlib import Path
import json, hashlib
import pandas as pd
root = Path.cwd()
manifest = json.loads((root/'ssa_run_manifest.json').read_text(encoding='utf-8'))
for name, sha in manifest['checksums'].items():
    assert hashlib.sha256((root/name).read_bytes()).hexdigest() == sha, name
from surveybridge.services.provenance import verify_project
projects = json.loads((root/'ssa_source_projects.json').read_text(encoding='utf-8'))
for project in projects:
    assert verify_project(project)
records = json.loads((root/'ssa_observations.json').read_text(encoding='utf-8'))
assert sorted(records, key=lambda r:r['source_row_id']) == sorted([r for p in projects for r in p['observations']], key=lambda r:r['source_row_id'])
print('All archived project lineages and exported checksums verified')

In [ ]:
summary = json.loads((root/'ssa_summary.json').read_text(encoding='utf-8'))
print(summary['completeness'])
print(summary['warnings'])
print(summary['availability_statuses'])
latest = pd.read_csv(root/'latest_survey_releases.csv')
display(latest)

In [ ]:
import matplotlib.pyplot as plt
counts = pd.Series(summary['country_totals']).sort_values()
if not counts.empty:
    ax = counts.plot.barh(figsize=(9, max(4, len(counts)*0.22)))
    ax.set_xlabel('Retrieved API observation records across requested indicators and survey rounds')
    ax.set_ylabel('DHS country code')
    ax.set_title('Data coverage — not a comparison of health outcomes')
    plt.tight_layout()
    plt.show()
# No report estimate extraction, annual interpolation, pooled indicators or significance tests.